In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import umap
from IPython.display import Image, display
from sklearn.neighbors import NearestNeighbors

working_dir = Path.cwd().resolve()
PROJECT_ROOT = working_dir.parent if working_dir.name == "figures" else working_dir
FIGURES_DIR = PROJECT_ROOT / "figures"
FIGUTIL_DIR = FIGURES_DIR / "figutil"
FIGDATA_DIR = FIGURES_DIR / "figdata/patient_umap"
CACHE_DIR = FIGDATA_DIR
FIGDATA_DIR.mkdir(parents=True, exist_ok=True)

for module_dir in (PROJECT_ROOT, FIGUTIL_DIR):
    if str(module_dir) not in sys.path:
        sys.path.insert(0, str(module_dir))

from data import Cohort, landmark
from plot_current_checkpoint_umap import (
    DEFAULT_CHECKPOINT,
    DEFAULT_DATA,
    extract_embeddings,
    render_figure,
    write_coordinates,
)
from utils import load_checkpoint, seed_everything

CHECKPOINT_PATH = DEFAULT_CHECKPOINT
DATA_PATH = DEFAULT_DATA
RECOMPUTE_FROM_CHECKPOINT = False
BATCH_SIZE = 256
MINIMUM_HISTORY_DAYS = 365
SEED = 42
N_NEIGHBORS = 30
MIN_DIST = 0.2

print(f"Project root: {PROJECT_ROOT}")
print(f"Checkpoint: {CHECKPOINT_PATH}")
print(f"Data: {DATA_PATH}")
print(f"Recompute from checkpoint: {RECOMPUTE_FROM_CHECKPOINT}")


## Data and representation extraction

기본 실행은 기존 artifact를 불러옵니다. 재계산 모드에서는 cohort landmark를 다시 선택하고 checkpoint의 final hidden state를 추출합니다.


In [ ]:
seed_everything(SEED)

cached_embedding_path = CACHE_DIR / "landmark_embeddings.npz"
cached_coordinate_path = CACHE_DIR / "umap_coordinates.csv.gz"

if RECOMPUTE_FROM_CHECKPOINT:
    device = "cuda:0" if torch.cuda.is_available() else "cpu"
    model, checkpoint = load_checkpoint(CHECKPOINT_PATH, device)
    cohort = Cohort(DATA_PATH)
    selected = []
    for patient_index in range(len(cohort)):
        end = landmark(cohort.patient(patient_index), MINIMUM_HISTORY_DAYS)
        if end is not None:
            selected.append((patient_index, int(end)))

    embeddings, metadata = extract_embeddings(
        model, cohort, selected, device, BATCH_SIZE
    )
    metadata_frame = pd.DataFrame(metadata)
    checkpoint_step = int(checkpoint["step"])
else:
    if not cached_embedding_path.is_file() or not cached_coordinate_path.is_file():
        raise FileNotFoundError(
            "Cached artifacts are missing. Set RECOMPUTE_FROM_CHECKPOINT=True."
        )
    cached = np.load(cached_embedding_path)
    embeddings = cached["embedding"].astype(np.float32, copy=False)
    cached_frame = pd.read_csv(cached_coordinate_path)
    metadata_frame = cached_frame.drop(columns=["umap_1", "umap_2"])
    checkpoint_step = 10_500

# Always refit UMAP so the saved coordinates and figure use SEED=9995.
reducer = umap.UMAP(
    n_components=2,
    n_neighbors=N_NEIGHBORS,
    min_dist=MIN_DIST,
    metric="cosine",
    random_state=SEED,
    n_jobs=1,
    low_memory=True,
)
coordinates = reducer.fit_transform(embeddings).astype(np.float32, copy=False)

assert embeddings.shape == (len(metadata_frame), 384)
assert coordinates.shape == (len(metadata_frame), 2)
assert np.isfinite(embeddings).all()
assert np.isfinite(coordinates).all()

print(f"Patients: {len(metadata_frame):,}")
print(f"Embeddings: {embeddings.shape}")
print(f"Coordinates: {coordinates.shape}")


## Patient UMAP

같은 UMAP 좌표에 density, sex, age, next observed event, T2DM history, medication history를 각각 색으로 표시합니다.


In [ ]:
metadata = metadata_frame.to_dict("records")
main_figure, main_png, main_pdf = render_figure(
    coordinates,
    metadata,
    FIGURES_DIR,
    checkpoint_step,
    MINIMUM_HISTORY_DAYS,
)

output_coordinate_path = FIGDATA_DIR / "umap_coordinates.csv.gz"
write_coordinates(output_coordinate_path, coordinates, metadata)

# Figure export is intentionally disabled until the final figure is approved.
# main_figure.savefig(main_png, dpi=300, facecolor="white")
# main_figure.savefig(main_pdf, facecolor="white")
display(main_figure)
plt.close(main_figure)
print("Rendered inline; figure files were not saved.")
print(f"Saved essential coordinates: {output_coordinate_path}")


In [ ]:
neighbor_model = NearestNeighbors(n_neighbors=16, metric="euclidean")
neighbor_indices = neighbor_model.fit(coordinates).kneighbors(return_distance=False)[:, 1:]
number_of_patients = len(metadata_frame)


def categorical_neighbor_score(column):
    labels = metadata_frame[column].astype(str).to_numpy()
    observed = float(np.mean(labels[:, None] == labels[neighbor_indices]))
    counts = pd.Series(labels).value_counts().to_numpy(dtype=float)
    baseline = float(np.sum(counts * (counts - 1)) / (number_of_patients * (number_of_patients - 1)))
    return observed, baseline, observed / baseline


rng = np.random.default_rng(SEED)
random_peer = rng.integers(0, number_of_patients, size=number_of_patients)

results = []
for column, label in [
    ("sex", "Same sex"),
    ("t2dm_history", "Same T2DM-history status"),
    ("next_event_type", "Same next-event type"),
]:
    observed, baseline, ratio = categorical_neighbor_score(column)
    results.append(
        {
            "diagnostic": label,
            "neighbor_value": observed,
            "random_baseline": baseline,
            "relative_ratio": ratio,
            "direction": "higher is more structured",
        }
    )

for column, label in [
    ("age_years", "Age absolute difference (years)"),
    ("prior_medication_classes", "Medication-class count absolute difference"),
]:
    values = metadata_frame[column].to_numpy(dtype=float)
    neighbor_difference = float(np.mean(np.abs(values[:, None] - values[neighbor_indices])))
    random_difference = float(np.mean(np.abs(values - values[random_peer])))
    results.append(
        {
            "diagnostic": label,
            "neighbor_value": neighbor_difference,
            "random_baseline": random_difference,
            "relative_ratio": neighbor_difference / random_difference,
            "direction": "lower is more structured",
        }
    )

diagnostic_table = pd.DataFrame(results)
display(diagnostic_table.round(3))


## Cohort summary


In [ ]:
cohort_summary = {
    "Patients": len(metadata_frame),
    "Median age": float(metadata_frame["age_years"].median()),
    "Recorded T2DM": int((metadata_frame["t2dm_history"] == "Recorded T2DM").sum()),
    "Any modeled medication history": int((metadata_frame["prior_medication_classes"] > 0).sum()),
    "Next event = disease": int((metadata_frame["next_event_type"] == "Disease").sum()),
    "Next event = medication": int((metadata_frame["next_event_type"] == "Medication").sum()),
    "No later event": int((metadata_frame["next_event_type"] == "No later event").sum()),
}
display(pd.Series(cohort_summary, name="value").to_frame())
